<span style="font-size:24px; font-family:'Roboto'; font-weight:bold;">
Script to make SHD files from CHD files.
</span><br>
iMOD doesn't accept nan values as SHD, but CHD files have nan (for cells where the layer thickness is 0, thus the model is inactive).<br>
We'll fill them with interpolated values. The values don't really matter.<br>
Some layers are inactive (not due to IBOUND, but cause Thk=0) in large parts of the model area. We'll fill (interpolate between, then extrapolate) the NaN values to the extend of L1 (which is active everywhere). Some layersare completely inactive. Those will be filled with values copied from the layer above.

This script can be improved when I have time. Then I should convert it to a .py file.

## 0.0. Libraries

In [7]:
from pathlib import Path
import imod
import xarray as xr
import numpy as np
from WS_Mdl.core import Mdl_N, Sep, sprint

In [1]:
import matplotlib.pyplot as plt

## 0.1. Options

In [2]:
MdlN_S = 'NBr54'
MdlN_B = 'NBr1'

In [20]:
date_B = '19951228'
date_S = '19960101'

In [15]:
M = Mdl_N(MdlN_S)

In [16]:
Pa_CHD = M.Pa.WS / fr"models\NBr\In\CHD\{MdlN_B}"
Pa_SHD = M.Pa.WS / fr"models\NBr\In\SHD\{MdlN_S}"

# 1. Read CHD, fill (interpolate), save as SHD 

In [21]:
l_CHD = list(Pa_CHD.glob(f'HEAD_{date_B}*.idf'))

In [27]:
DA_CHD = imod.formats.idf.open(l_CHD, pattern=f'{{name}}_{date_B}_L{{layer}}_NBr1')

In [28]:
DA_CHD

<xarray.DataArray 'head' (layer: 37, y: 280, x: 360)> Size: 30MB
dask.array<stack, shape=(37, 280, 360), dtype=float64, chunksize=(1, 280, 360), chunktype=numpy.ndarray>
Coordinates:
  * layer    (layer) int64 296B 1 2 3 4 5 6 7 8 9 ... 29 30 31 32 33 34 35 36 37
  * y        (y) float64 2kB 4.199e+05 4.196e+05 ... 3.504e+05 3.501e+05
  * x        (x) float64 3kB 7.012e+04 7.038e+04 ... 1.596e+05 1.599e+05
    dx       float64 8B 250.0
    dy       float64 8B -250.0

In [29]:
# 1. Sort coordinates to allow interpolation
reversed_y = not np.all(np.diff(DA_CHD.y.values) > 0)
reversed_x = not np.all(np.diff(DA_CHD.x.values) > 0)

In [30]:
if reversed_y:
    DA_CHD = DA_CHD.sortby("y")
if reversed_x:
    DA_CHD = DA_CHD.sortby("x")

In [31]:
# fig, ax = plt.subplots()
# img = ax.imshow(mask_valid, cmap="gray")
# cbar = fig.colorbar(img, ax=ax)

In [32]:
mask_valid = ~DA_CHD.isel(layer=0).isnull() # Get valid mask from layer 0

In [33]:
# 2. Fill missing values with inter/extrapolation.
DA_CHD_interp_list = []
for i in range(DA_CHD.sizes["layer"]):
    layer_i = DA_CHD.isel(layer=i)
    
    if layer_i.isnull().all():
        filled = DA_CHD_interp_list[-1]  # if the layer is all NaN use previous L
    else:
        filled = (
            layer_i.interpolate_na(dim="y", method="linear")
                    .interpolate_na(dim="x", method="linear")
                    .fillna(layer_i.ffill("y").bfill("y").ffill("x").bfill("x"))
                    .where(mask_valid)
        )
    DA_CHD_interp_list.append(filled.assign_coords(layer=DA_CHD.layer.isel(layer=i)))

DA_CHD_interp = xr.concat(DA_CHD_interp_list, dim="layer")
DA_CHD_interp["layer"] = DA_CHD["layer"]

In [34]:
# A = DA_CHD_interp.isel(layer=1-1).squeeze().sortby('y', ascending=False)
# fig, ax = plt.subplots()
# img = ax.imshow(A, cmap="inferno")
# cbar = fig.colorbar(img, ax=ax)

In [35]:
# 3. Reverse coords back to original orientation
if reversed_y:
    DA_CHD_interp = DA_CHD_interp.sortby("y", ascending=False)
if reversed_x:
    DA_CHD_interp = DA_CHD_interp.sortby("x", ascending=False)

In [36]:
DA_CHD_interp = DA_CHD_interp.expand_dims(name=[f"SHD_{date_S}"])

In [41]:
imod.idf.save(Pa_SHD / "dummy.idf", DA_CHD_interp, pattern=f"{{name}}_L{{layer}}_{MdlN_S}.IDF")

# 2. Write SHD block

In [44]:
for i in range(37):
    print(fr" 1,2, {i+1:03},   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\{MdlN_S}\SHD_20100114_L{i+1}_{MdlN_S}.IDF' >>> (shd) starting heads (idf) <<<")

 1,2, 001,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L1_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 002,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L2_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 003,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L3_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 004,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L4_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 005,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L5_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 006,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L6_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 007,   1.000000    ,   0.000000    ,  -999.9900    , '..\..\In\SHD\NBr54\SHD_20100114_L7_NBr54.IDF' >>> (shd) starting heads (idf) <<<
 1,2, 008,   

# 3. Write metadata file in the same folder

In [43]:
with open(Pa_SHD/ '_metadata.txt    ', 'w') as f:
    f.write(f"This file was produced by 'G:\code\PrP\CHD_to_SHD\CHD_to_SHD_{MdlN_S}.ipynb' cause significant differences were spotted between the CHD (of the 1st SP) and SHDs of NBr38.)")

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Karam014\AppData\Local\Temp\ipykernel_27952\152742519.py:2: SyntaxWarning: invalid escape sequence '\c'
  f.write(f"This file was produced by 'G:\code\PrP\CHD_to_SHD\CHD_to_SHD_{MdlN_S}.ipynb' cause significant differences were spotted between the CHD (of the 1st SP) and SHDs of NBr38.)")
